# Data Preparation

- [Data Cleaning and Feature Selection](#Data-Cleaning-and-Feature-Selection)
  - [Missing data and missingness mechanisms](#Missing-data-and-missingness-mechanisms)
  - [Outliers](#Outliers)
  - [Feature scaling](#Feature-scaling)
  - [Multicollinearity](#Multicollinearity)
  - [Feature selection: filter, wrapper and embedded methods](#Feature-selection:-filter,-wrapper-and-embedded-methods)
  - [Data leakage](#Data-leakage)
- [Feature Engineering](#Feature-Engineering)
  - [Core idea](#Core-idea)
  - [Categorical encoding](#Categorical-encoding)
  - [Numerical transforms](#Numerical-transforms)
  - [Interaction features](#Interaction-features)
  - [Domain-specific and LLM-derived features](#Domain-specific-and-LLM-derived-features)
  - [Cyclical (periodic) features](#Cyclical-(periodic)-features)
- [Text Features](#Text-Features)
  - [Stemming vs. Lemmatization](#Stemming-vs.-Lemmatization)
  - [N-grams](#N-grams)
  - [The problem TF-IDF solves](#The-problem-TF-IDF-solves)
  - [Term Frequency (TF)](#Term-Frequency-(TF))
  - [Inverse Document Frequency (IDF)](#Inverse-Document-Frequency-(IDF))
  - [Combining TF × IDF — the payoff](#Combining-TF-×-IDF-—-the-payoff)
  - [L2 normalization](#L2-normalization)
  - [Comparing against sklearn's `TfidfVectorizer`](#Comparing-against-sklearn's-TfidfVectorizer)
  - [Limitations and when to use TF-IDF vs. embeddings](#Limitations-and-when-to-use-TF-IDF-vs.-embeddings)
- [Imbalanced Data](#Imbalanced-Data)
  - [Random Oversampling and Undersampling](#Random-Oversampling-and-Undersampling)
  - [SMOTE (Synthetic Minority Oversampling Technique)](#SMOTE-(Synthetic-Minority-Oversampling-Technique))

### Data Cleaning and Feature Selection

#### Missing data and missingness mechanisms

Deletion (if MCAR, small %), mean/median/mode imputation (simple, biases variance), model-based imputation (KNN/MICE, more accurate), or flag-as-category (tree models can use missingness itself as signal).

MNAR — missingness depends on the missing value itself, so mean/median imputation biases results; model the mechanism or run sensitivity analysis and add an indicator. MCAR: unrelated to any data — deletion unbiased (just less data). MAR: depends on observed variables — impute using them (MICE, model-based). MNAR: depends on the missing value itself — imputation biased; model the missingness / sensitivity analysis. Add a missing-indicator; test missingness vs other columns.

#### Outliers

IQR rule (below Q1-1.5×IQR or above Q3+1.5×IQR), z-score (|z|>3), or model-based (Isolation Forest, LOF). Choice depends on distribution shape/dimensionality.

#### Feature scaling

Standardization (x−μ)/σ: zero mean/unit variance — default for linear models, SVM, NN, PCA. Min-max: [0,1], outlier-sensitive (images/NN/KNN). Needed for distance- and gradient-based models; NOT for trees. Fit the scaler on train only (leakage).

#### Multicollinearity

VIF>5-10 signals problematic collinearity between predictors. Fixes: drop/combine correlated features, PCA, or use L2/Ridge to stabilize coefficients.

#### Feature selection: filter, wrapper and embedded methods

Filter: statistical score independent of model (correlation, chi-square, mutual info) — fast, model-agnostic. Wrapper: search subsets via model performance (RFE) — accurate, expensive. Embedded: built into training (Lasso coefs, tree importances) — efficient middle ground.

#### Data leakage

Information from outside the training window (future data, target-derived feature, or eval-set contamination) leaks into training, inflating offline metrics. Fix: strict time-ordering, fit all preprocessing/resampling on train fold only.

### Feature Engineering

#### Core idea

Transform raw data into a representation that exposes patterns to a model more directly than the raw values do. The same underlying information, reshaped so the model does not have to work as hard (or cannot work hard enough with a simple model like logreg) to find the pattern itself.

This notebook covers categorical encoding, numerical transforms, interaction features, and text/date features. Cyclical (sin/cos) encoding for time features is covered in the Cyclical (periodic) features section below, not repeated here.

#### Categorical encoding

Toy: `contact_channel` column with values [dating app, email, dating app, social media DM].

One-hot encoding: one binary column per category.
```
              dating_app  email  social_media_DM
dating app        1         0          0
email              0         1          0
dating app        1         0          0
social media DM    0         0          1
```
Simple, no ordering implied, but dimensionality explodes with high-cardinality categories (this project's `impersonated_entity` field, free text, could have dozens of distinct values).

Label/ordinal encoding: one integer per category (dating app=0, email=1, social media DM=2). Compact, but implies a false ordering, a linear model would treat "social media DM" (2) as literally twice "email" (1), meaningless for a truly unordered category. Fine for tree models, which just split on thresholds and do not assume the numbers are meaningfully ordered.

Target/mean encoding: replace each category with the average target value for that category. Powerful (captures the actual relationship, not just an arbitrary label), but leaks the target into the feature if computed carelessly. The CatBoost notebook in this series works through this leakage problem and its fix (Ordered Target Statistics) in full detail with a worked example, same underlying issue as any naive target-mean encoding here.

Frequency encoding: replace each category with how often it appears in the data. Cheap, no leakage risk, loses information about the relationship to the target though, purely about prevalence.

In [ ]:
import pandas as pd

df = pd.DataFrame({"contact_channel": ["dating app", "email", "dating app", "social media DM"]})

print("one-hot:\n", pd.get_dummies(df["contact_channel"]))

print("\nlabel encoding:\n", df["contact_channel"].astype("category").cat.codes)

print("\nfrequency encoding:\n", df["contact_channel"].map(df["contact_channel"].value_counts()))

#### Numerical transforms

Standardization (z-score): (x - mean) / std. Needed for distance-based (KNN, k-means, SVM) and gradient-based (logreg, neural nets) models, not needed for trees, covered with the reasoning already in the KNN and PCA notebooks.

Log transform for skewed distributions. Worked example, amount_lost has values [500, 2500, 10000, 40000], a heavily right-skewed distribution (a few huge values dominate the range):
```
raw values:        [500, 2500, 10000, 40000]
log(raw values):    [6.21, 7.82, 9.21, 10.60]
```
The raw range spans 39,500 (500 to 40000), the log range spans only 4.4 (6.21 to 10.6), the log-transformed feature is far more evenly spread, which helps linear models in particular (a coefficient can now capture "each unit of log-amount matters equally" rather than being dominated by the handful of largest raw values).

Binning: convert a continuous feature into discrete ranges, e.g. amount_lost into [under $1000, $1000-$10000, over $10000]. Trades precision for robustness to outliers and for letting a linear model capture non-linear thresholds it otherwise could not (a single linear coefficient cannot represent "risk jumps at $10000," a binned indicator can).

In [ ]:
import numpy as np

amounts = np.array([500, 2500, 10000, 40000])
log_amounts = np.log(amounts)

print("raw range:", amounts.max() - amounts.min())
print("log range:", (log_amounts.max() - log_amounts.min()).round(2))
print("log values:", log_amounts.round(2))

bins = pd.cut(amounts, bins=[0, 1000, 10000, np.inf], labels=["under_1k", "1k_to_10k", "over_10k"])
print("\nbinned:", list(bins))

#### Interaction features

A single feature might not carry signal alone, but combined with another one it does. Worked example: `urgency_language` (0/1) alone, and `payment_method_requested == gift card` (0/1) alone, might each be weak signals individually, but their interaction (urgency AND gift card requested together) is a much stronger fraud signal than either alone, that specific combination is a very typical high-pressure scam pattern.
```
urgency=1, gift_card=1  -> interaction feature = 1  (both present, strong signal)
urgency=1, gift_card=0  -> interaction feature = 0
urgency=0, gift_card=1  -> interaction feature = 0
urgency=0, gift_card=0  -> interaction feature = 0
```
Linear models (logreg) cannot discover this combination on their own, a single weighted sum of the two features separately cannot represent "only matters when both are true together," you have to engineer the interaction term explicitly and hand it a new column. Tree-based models (the XGBoost, LightGBM, CatBoost notebooks) can discover interactions like this automatically through nested splits, one reason they often need less manual interaction-feature engineering than linear models do.

#### Domain-specific and LLM-derived features

Feature engineering is not always mechanical transforms of existing columns, it can involve genuinely new information extracted from unstructured data. The fraud project's `FEATURE_SCHEMA` (contact_channel, impersonated_entity, payment_method_requested, urgency_language, victim_sent_money) is exactly this: an LLM reads unstructured narrative text and extracts structured features that did not exist as columns before, a form of feature engineering that only became practical with LLMs, previously this kind of signal required manual labeling or complex NLP pipelines (named entity recognition, rule-based extraction) to get anywhere close.

In [ ]:
df_interact = pd.DataFrame({
    "urgency": [1, 1, 0, 0],
    "gift_card": [1, 0, 1, 0],
})
df_interact["urgency_and_giftcard"] = df_interact["urgency"] * df_interact["gift_card"]
print(df_interact)

#### Cyclical (periodic) features

In [ ]:
import numpy as np
import pandas as pd

# Create sample data
hours = [0, 6, 12, 18, 23, 24]  # 24 wraps to 0
df = pd.DataFrame({'hour': hours})

# Encode cyclically
df['sin_hour'] = np.sin(df['hour'] * 2 * np.pi / 24)
df['cos_hour'] = np.cos(df['hour'] * 2 * np.pi / 24)

print(df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hours = np.arange(24)
sin_hours = np.sin(hours * 2 * np.pi / 24)
cos_hours = np.cos(hours * 2 * np.pi / 24)

plt.figure(figsize=(6,6))
plt.scatter(sin_hours, cos_hours, c=hours, cmap='hsv')
plt.xlabel('sin(hour)')
plt.ylabel('cos(hour)')
plt.title('Cyclical Encoding of Hours')
plt.axis('equal')
for i, h in enumerate(hours):
    plt.annotate(str(h), (sin_hours[i], cos_hours[i]))
plt.show()

### Text Features

#### Stemming vs. Lemmatization

Both reduce a word to a base form, so "running", "runs", "ran" collapse to one feature instead of three separate ones. Different mechanisms:

Stemming: crude, rule-based suffix stripping, fast but can produce non-words. Worked example: "studies" -> "studi" (strips "es", does not know this leaves an invalid word), "studying" -> "study".

Lemmatization: dictionary-aware, looks up the actual base (dictionary) form, slower but always produces a real word, and is context/part-of-speech aware. "studies" -> "study" (correct dictionary form), "better" -> "good" (lemmatization understands irregular forms, stemming has no mechanism for this at all, it only strips suffixes).

In [ ]:
# real from-scratch stemmer, crude rule-based suffix stripping, demonstrates the actual mechanism
def simple_stem(word):
    suffixes = ["ing", "ed", "es", "s"]
    for suf in sorted(suffixes, key=len, reverse=True):  # try longest suffix first
        if word.endswith(suf) and len(word) - len(suf) >= 2:
            return word[:-len(suf)]
    return word

for word in ["studies", "studying", "runs", "better"]:
    print(f"{word} -> stem: {simple_stem(word)}")

print("\n'better' is unchanged, no suffix to strip, this crude stemmer has zero mechanism")
print("for irregular forms, exactly the gap a real dictionary-based lemmatizer fills.")
print("(nltk's PorterStemmer/WordNetLemmatizer are the production version of this idea,")
print("pip install nltk to use them, not installed in this environment)")

#### N-grams

TF-IDF and bag-of-words treat every word independently, losing word order entirely (flagged as a limitation in `04-data-prep.ipynb`). N-grams partially recover order by treating contiguous word SEQUENCES as the unit instead of single words.

Worked example, sentence "not a scam":
```
unigrams (1-grams): ["not", "a", "scam"]
bigrams (2-grams):  ["not a", "a scam"]
```
Unigram "scam" alone cannot distinguish "a scam" from "not a scam", both contain the word "scam" with identical weight. The bigram "not a" captures the negation context that unigrams throw away entirely. Cost: vocabulary size grows fast (every new bigram is a new column), and even bigrams cannot capture negation across a longer gap ("not, in my opinion, a scam").

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ["not a scam", "definitely a scam"]
vectorizer = CountVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(docs)
print("vocabulary:", vectorizer.get_feature_names_out())
print(X.toarray())

#### The problem TF-IDF solves

The simplest way to turn text into numbers is a **bag-of-words** count: one
column per vocabulary word, value = how many times that word appears in the
document. Problem: this treats every word as equally important. A word like
"the" appears constantly in every document and tells you nothing about what
that document is *about*, but a raw count would rank it as the most
"important" word in almost any English document, just by frequency.

TF-IDF fixes this with one idea: **a word matters to a document if it's
frequent in that document AND rare across the rest of the corpus.** Frequent
+ rare-elsewhere = distinctive. Frequent + common-everywhere = filler.

Worked corpus we'll use throughout this notebook — 3 tiny documents:
```
doc1: "the cat sat on the mat"
doc2: "the dog sat on the log"
doc3: "cats and dogs are great pets"
```
Notice "the" appears in doc1 and doc2 (common), while "cat"/"mat" appear only
in doc1 (distinctive to it). TF-IDF should end up ranking "cat"/"mat" as more
important to doc1 than "the" is — even though "the" appears MORE often within
doc1 itself. Keep that specific comparison in mind; we'll verify it numerically below.

#### Term Frequency (TF)

Formula: `tf(t, d) = count(t in d) / total_words(d)` — normalized by document
length, so a long document doesn't automatically get bigger TF values just
from having more words overall.

Worked example, doc1 = "the cat sat on the mat" (6 words total):
```
word counts: the=2, cat=1, sat=1, on=1, mat=1
tf(the, doc1) = 2/6 = 0.333
tf(cat, doc1) = 1/6 = 0.167
tf(sat, doc1) = 1/6 = 0.167
tf(on,  doc1) = 1/6 = 0.167
tf(mat, doc1) = 1/6 = 0.167
```
Notice: by raw TF alone, "the" already looks twice as important as any other
word in doc1 (0.333 vs 0.167) — this is the problem TF-IDF's IDF term exists
to correct. TF alone can't distinguish "the" from "cat"; both are just
"a word that appears in this document."

In [ ]:
from collections import Counter

corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]
docs = [d.split() for d in corpus]

def tf(word, doc):
    counts = Counter(doc)
    return counts[word] / len(doc)

# reproduce the hand-worked doc1 numbers
for word in ["the", "cat", "sat", "on", "mat"]:
    print(f"tf({word}, doc1) = {tf(word, docs[0]):.3f}")

#### Inverse Document Frequency (IDF)

Formula: `idf(t) = log(N / df(t))` — N = total documents in the corpus,
df(t) = number of documents containing term t at least once (not how many
times, just whether it appears at all).

Worked example, N=3 documents:
```
"the": appears in doc1, doc2 → df=2 → idf = log(3/2) = log(1.5) ≈ 0.405
"cat": appears in doc1 only  → df=1 → idf = log(3/1) = log(3)   ≈ 1.099
"sat": appears in doc1, doc2 → df=2 → idf ≈ 0.405
"on":  appears in doc1, doc2 → df=2 → idf ≈ 0.405
"mat": appears in doc1 only  → df=1 → idf ≈ 1.099
```
Already visible: "the" gets a LOWER idf (0.405) than "cat"/"mat" (1.099) —
common-across-documents words get penalized, rare-and-specific words get
boosted. This is the correction TF alone couldn't provide.

Why the log, specifically — without it, IDF would be a raw ratio `N/df(t)`.
For a word appearing in just 1 of 1000 documents, that ratio is 1000, while
a word in 500 of 1000 is only 2 — the rare word would dominate the score by
a factor of 500, wildly overweighting extremely rare (sometimes just noisy
or misspelled) terms. The log compresses that: log(1000)≈6.9 vs log(2)≈0.7,
a ratio of ~10x instead of 500x — rare terms still get more weight, but not
absurdly more.

In [ ]:
import math

def document_frequency(word, all_docs):
    return sum(1 for doc in all_docs if word in doc)

def idf(word, all_docs):
    N = len(all_docs)
    df = document_frequency(word, all_docs)
    return math.log(N / df)

for word in ["the", "cat", "sat", "on", "mat"]:
    print(f"idf({word}) = {idf(word, docs):.3f}")

#### Combining TF × IDF — the payoff

Formula: `tfidf(t, d) = tf(t, d) × idf(t)`

Worked example, doc1's full vector (multiplying the TF values from section 1
by the IDF values from section 2):
```
tfidf(the, doc1) = 0.333 × 0.405 = 0.135
tfidf(cat, doc1) = 0.167 × 1.099 = 0.183
tfidf(sat, doc1) = 0.167 × 0.405 = 0.068
tfidf(on,  doc1) = 0.167 × 0.405 = 0.068
tfidf(mat, doc1) = 0.167 × 1.099 = 0.183
```
**This is the exact thing we set out to verify in section 0**: "the" had the
HIGHEST raw term frequency in doc1 (0.333, twice any other word), but ends up
with a LOWER final TF-IDF score (0.135) than "cat" or "mat" (0.183) — because
"the" appears everywhere (low IDF), while "cat"/"mat" are specific to this
document (high IDF). The combination correctly demotes the frequent-but-
uninformative word and promotes the rarer-but-distinctive ones, exactly the
behavior a raw bag-of-words count could never produce.

In [ ]:
def tfidf(word, doc, all_docs):
    return tf(word, doc) * idf(word, all_docs)

doc1_vector = {word: tfidf(word, docs[0], docs) for word in set(docs[0])}
for word, score in sorted(doc1_vector.items(), key=lambda x: -x[1]):
    print(f"tfidf({word}, doc1) = {score:.3f}")

#### L2 normalization

Why: a longer document naturally accumulates a bigger raw TF-IDF vector
(more nonzero entries, more total "mass") even if it's not actually more
relevant to any given topic — without correcting for this, comparing two
documents' vectors (e.g. via dot product / cosine similarity) would be
biased toward whichever one is simply longer. L2-normalizing divides each
document's vector by its own magnitude, so every document vector has length
1 — comparisons become about the *direction* (which words matter, in what
proportion) rather than raw magnitude.

Formula: `v_normalized = v / ||v||₂`, where `||v||₂ = √(Σ vᵢ²)`

Worked example, doc1's vector from section 3 (only nonzero entries):
```
v = [the=0.135, cat=0.183, sat=0.068, on=0.068, mat=0.183]

||v||₂ = √(0.135² + 0.183² + 0.068² + 0.068² + 0.183²)
       = √(0.0182 + 0.0335 + 0.0046 + 0.0046 + 0.0335)
       = √0.0944 ≈ 0.307

normalized:
  the = 0.135/0.307 ≈ 0.439
  cat = 0.183/0.307 ≈ 0.596
  sat = 0.068/0.307 ≈ 0.221
  on  = 0.068/0.307 ≈ 0.221
  mat = 0.183/0.307 ≈ 0.596
```
Check: √(0.439² + 0.596² + 0.221² + 0.221² + 0.596²) ≈ √(0.193+0.355+0.049+0.049+0.355) = √1.0 = 1.0 ✓
The relative ordering (cat/mat highest, the middling, sat/on lowest) is
unchanged by normalization — only the overall scale shifts, which is exactly
the point: normalization doesn't change *what* the vector says about the
document, just puts it on a comparable footing with every other document's vector.

In [ ]:
import numpy as np

def l2_normalize(vector_dict):
    values = np.array(list(vector_dict.values()))
    norm = np.sqrt(np.sum(values ** 2))
    return {k: v / norm for k, v in vector_dict.items()}

doc1_normalized = l2_normalize(doc1_vector)
for word, score in sorted(doc1_normalized.items(), key=lambda x: -x[1]):
    print(f"{word}: {score:.3f}")

# sanity check: normalized vector's own L2 norm should be 1.0
check = np.sqrt(sum(v**2 for v in doc1_normalized.values()))
print("\nnorm of normalized vector:", check)

#### Comparing against sklearn's `TfidfVectorizer`

Important nuance: sklearn's default IDF formula is NOT the raw
`log(N/df)` used above — it's **smoothed**: `idf(t) = log((1+N)/(1+df(t))) + 1`.
The `+1` inside the log prevents division issues and avoids any term ever
reaching `df=N` and getting idf=0 (which raw log(N/df) would give a word
appearing in every document — technically correct, but sklearn treats
"appears everywhere" as still worth a small non-zero weight, not zero).
The trailing `+1` outside the log shifts every score up slightly.

Because of this, sklearn's numbers won't match our from-scratch numbers
exactly — that's expected, not a bug. What SHOULD match: the *relative
ordering* (cat/mat still outrank "the" after normalization) and the general
shape of the result.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), index=["doc1", "doc2", "doc3"])
print(df.round(3))

print("\ndoc1, sorted by score (compare ordering to our from-scratch result):")
print(df.loc["doc1"].sort_values(ascending=False).head(5))

#### Limitations and when to use TF-IDF vs. embeddings

- Ignores word order entirely — "dog bites man" and "man bites dog" produce
  the identical bag-of-words vector, despite opposite meanings.
- No notion of synonymy or semantic similarity — "car" and "automobile" are
  completely different columns, zero shared signal, even though they mean
  nearly the same thing. Dense embeddings (word2vec, sentence transformers)
  solve this by placing semantically similar words near each other in
  vector space; TF-IDF has no such geometry.
- Vocabulary is fixed at fit time — a word never seen during `fit` simply
  contributes nothing at inference (see the fraud project's baseline cell
  for exactly this: `vectorizer.transform(val)` uses train's vocabulary only).
- Where it still wins: interpretability (every dimension is literally a
  word, you can read off exactly which words drove a prediction), no
  training required beyond counting, and strong performance specifically
  when the task is close to keyword matching — the fraud project's own
  result (logreg + TF-IDF beat every tree-based model, including hybrid
  LLM-feature setups) is a real example of TF-IDF's strength on a task
  where vocabulary alone is highly discriminative.

### Imbalanced Data

A data-level fix for class imbalance, distinct from focal loss's loss-level fix (see `05-loss-regularization.ipynb`'s Focal loss section). Covers random oversampling, random undersampling, SMOTE.

#### Random Oversampling and Undersampling

Toy setup: 8 majority-class examples, 2 minority-class examples, labels = [0]*8 + [1]*2.

Random oversampling: duplicate minority-class rows (with replacement) until the classes are balanced, or closer to it.
```
original: 8 class-0, 2 class-1
oversample class-1 by duplicating: 8 class-0, 8 class-1 (the 2 real rows each appear 4 times)
```
Downside: no new information, the model sees literally the same 2 minority rows repeated, real risk of overfitting to those 2 specific examples, since gradient-based models effectively see them 4x as often in every batch.

Random undersampling: drop majority-class rows until balanced.
```
original: 8 class-0, 2 class-1
undersample class-0 by dropping 6 of them: 2 class-0, 2 class-1
```
Downside: throws away 75% of the majority class data, potentially real, useful signal, especially painful when the majority class is not actually huge in absolute terms.

In [ ]:
import numpy as np

X = np.arange(10).reshape(-1, 1)
y = np.array([0]*8 + [1]*2)

# oversample: duplicate minority rows
minority_idx = np.where(y == 1)[0]
oversample_idx = np.concatenate([np.arange(10), np.tile(minority_idx, 3)])  # 3 extra copies each
print("after oversampling, class counts:", np.bincount(y[oversample_idx]))

# undersample: drop majority rows
majority_idx = np.where(y == 0)[0]
rng = np.random.default_rng(0)
keep_majority = rng.choice(majority_idx, size=2, replace=False)
undersample_idx = np.concatenate([keep_majority, minority_idx])
print("after undersampling, class counts:", np.bincount(y[undersample_idx]))

#### SMOTE (Synthetic Minority Oversampling Technique)

Fixes random oversampling's "no new information" problem by generating genuinely NEW synthetic minority points, interpolated between real minority points and their nearest minority neighbors, rather than copying existing rows.

Formula: new_point = point_a + r * (point_b - point_a), where point_b is one of point_a's nearest minority-class neighbors, and r is a random fraction between 0 and 1.

Worked example, 2D, two real minority points that are nearest neighbors of each other: point_a=(1,1), point_b=(3,3). Generate one synthetic point with r=0.5:
```
new_point = (1,1) + 0.5 * ((3,3) - (1,1))
          = (1,1) + 0.5 * (2,2)
          = (1,1) + (1,1)
          = (2,2)
```
(2,2) is a genuinely new point, sitting exactly halfway along the line segment between the two real minority points, not a duplicate of either. A different random r would land somewhere else on that same segment, e.g. r=0.2 gives (1.4, 1.4), r=0.8 gives (2.6, 2.6).

In [ ]:
import numpy as np

point_a = np.array([1, 1])
point_b = np.array([3, 3])

for r in [0.2, 0.5, 0.8]:
    synthetic = point_a + r * (point_b - point_a)
    print(f"r={r}: synthetic point = {synthetic}")

**Practical notes.**

Still has a real limitation: interpolating between minority points only makes sense if the minority class occupies a genuinely coherent, continuous region of feature space, if the minority class is actually several distinct sub-patterns (e.g. several genuinely different fraud typologies lumped under one "fraud" label), SMOTE can generate synthetic points that fall in the gap BETWEEN two real sub-patterns, a region that does not represent any real minority behavior at all. Variants exist to address this (Borderline-SMOTE focuses synthesis near the decision boundary specifically, ADASYN adapts the number of synthetic points generated per region based on how hard that region is to classify), not covered in depth here.

Applied only to the TRAINING set, never to validation/test, synthesizing fake examples into your evaluation data would let the model be "tested" on interpolated points that never occurred in reality, an evaluation-side leakage equivalent to the encoding leakage covered in the CatBoost notebook, just a different mechanism.

In [ ]:
# requires: pip install imbalanced-learn (not installed in this environment)
# from imblearn.over_sampling import SMOTE
#
# smote = SMOTE(k_neighbors=1, random_state=42)
# X_resampled, y_resampled = smote.fit_resample(X, y)
# print("class counts after SMOTE:", np.bincount(y_resampled))
print("see commented block above, requires: pip install imbalanced-learn")